# Super-résolution GWS — FUSION + HADA + Hydro-FE + Feature Selection (sans XAI)

Notebook Google Colab généré à partir du script Python d'origine :

**FUSION + SR_GWS + HADA + Hydro-FE (Hydro-Attention D8) + Feature Selection (TabNet stable + mRMR/group)**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files_SR.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Les résultats sont écrits dans `input_data_ML/SR_GWS_FUSION_HADA_HYDROFE_FS_XAI_HYDROINV_MULTI` sur Drive.

**Version sans XAI :** les cartes Hydro-XAI, l'incertitude par arbre et HydroINV ont été retirées. Le notebook produit les rasters SR (grille fine), les prédictions coarse et les fichiers de sélection de variables.

Les lignes ajoutées ou modifiées pour Colab sont repérées par `# [COLAB]`, les retraits liés à la XAI par `# [SANS XAI]`.

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab. `pytorch-tabnet` est optionnel : s'il est absent,
le classement des variables bascule automatiquement sur l'importance par permutation.

In [ ]:
!pip install -q rasterio pytorch-tabnet

### A.3 Imports

In [ ]:
import os
import re
import time
import glob
import shutil
from datetime import datetime, date, timedelta

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Optional permutation importance
try:
    from sklearn.inspection import permutation_importance
    _HAS_PERM_IMPORTANCE = True
except Exception:
    permutation_importance = None
    _HAS_PERM_IMPORTANCE = False

# Optional TabNet
try:
    from pytorch_tabnet.tab_model import TabNetRegressor
    _HAS_TABNET = True
except Exception:
    TabNetRegressor = None
    _HAS_TABNET = False

# [COLAB] Bilan des dépendances optionnelles
print("rasterio :", rasterio.__version__)
print("TabNet disponible :", _HAS_TABNET, "| permutation_importance :", _HAS_PERM_IMPORTANCE)

## 0. Paramètres

### 0. Paramètres généraux (chemins, dates)

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# =============================================================================
# 0) PARAMETRES GENERAUX
# =============================================================================
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files_SR.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon les dossiers de sortie seraient créés hors de Drive et perdus en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

OUT_DIR = os.path.join(BASE_DIR, "SR_GWS_FUSION_HADA_HYDROFE_FS_XAI_HYDROINV_MULTI")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_NODATA = -9999.0

# Split dates
TEST_FRACTION = 0.10
RANDOM_SEED = 42

# Fenêtres (si vous voulez contraindre temporellement)
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = date(2025, 7, 1)
FORECAST_END   = date(2025, 10, 30)

# Dates exclues
EXCLUDED_DATES = set()

### 0bis. Sections attendues et fusion des entrées

In [ ]:
# =============================================================================
# 0bis) SECTIONS ATTENDUES (SR)
# =============================================================================
SECTION_CHIRPS = "CHIRPS_Clipped"      # fine (template sortie)
SECTION_API    = "Precipitations_API"  # fine
SECTION_GWS    = "Total_GWS"           # coarse (cible)
SECTION_RZSM   = "SMAP_RZSM_Clipped"   # optionnel (fine)

SECTION_SLOPE  = "Slope"               # statique
SECTION_DTM    = "MNT"                 # statique DEM
SECTION_TWI    = "TWI"                 # statique
SECTION_DTR    = "Distance_To_River"   # statique

USE_SIN_COS_DOY = True

# =============================================================================
# 0bis-A) FUSION INPUTS (anciens + nouveaux) -> BASE_DIR/_FUSED/...
# =============================================================================
FUSION_ENABLE = True
FUSION_ROOT = os.path.join(BASE_DIR, "_FUSED")

# conflit par date: "PREFER_LAST" (dernière source listée gagne) ou "PREFER_FIRST"
FUSION_CONFLICT_POLICY = "PREFER_LAST"

# écraser si un fichier fusionné existe déjà
FUSION_OVERWRITE = True

# sections qui doivent être fusionnées (dyn + statics)
FUSION_SECTIONS = [
    SECTION_CHIRPS, SECTION_API, SECTION_GWS, SECTION_RZSM,
    SECTION_SLOPE, SECTION_DTM, SECTION_TWI, SECTION_DTR
]

### 0ter / 0.1. HADA (Hydro-Aware Dual Attention) et Hydro-FE (D8 + softmax)

In [ ]:
# =============================================================================
# 0ter) HADA (Hydro-Aware Dual Attention)
# =============================================================================
HADA_ENABLE = True

HADA_LAM = 3.0
HADA_A0  = -1.0
HADA_A1  =  2.0   # TWI
HADA_A2  =  2.0   # corridor rivière (DTR)
HADA_A3  =  1.0   # API
HADA_A4  =  0.5   # CHIRPS
HADA_A5  =  1.0   # SLOPE (soustrait)

HADA_GAMMA = 0.7   # gating: x' = x*(1+gamma*att)
HADA_WMIN = 0.5
HADA_WMAX = 2.0

# =============================================================================
# 0.1) HYDRO-FE (D8 + softmax) + Hydro Attention
# =============================================================================
FE_ENABLE = True
FE_HOPS = 8

FE_BETA_ATT = 2.0
FE_DZ_W     = 1.0
FE_SOFT_EPS = 1e-8
FE_HOP_DECAY = 0.90

FE_ATT_USE_GRAD = True
FE_ATT_GRAD_W   = 1.0
FE_ATT_BETA = 1.5
FE_ATT_CLIP = 20.0

### 0.2. Régularisation RF et sélection de variables (TabNet stable)

In [ ]:
# =============================================================================
# 0.2) RF REGULARISATION + FS TABNET STABLE
# =============================================================================
REG_MODE = "AUTO"  # "AUTO" ou "FIXED"

RF_BASE_PARAMS = {
    "n_estimators": 800,
    "bootstrap": True,
}

RF_FIXED_REG_PARAMS = {
    "max_depth": 25,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": 0.4,
}

RF_REG_CANDIDATES = [
    {"max_depth": 30, "min_samples_split": 20, "min_samples_leaf": 10, "max_features": 0.5,  "max_samples": 0.9},
    {"max_depth": 25, "min_samples_split": 20, "min_samples_leaf": 10, "max_features": 0.4,  "max_samples": 0.8},
    {"max_depth": 25, "min_samples_split": 30, "min_samples_leaf": 15, "max_features": 0.4,  "max_samples": 0.8},
    {"max_depth": 20, "min_samples_split": 30, "min_samples_leaf": 20, "max_features": 0.35, "max_samples": 0.7},
    {"max_depth": 18, "min_samples_split": 40, "min_samples_leaf": 25, "max_features": 0.30, "max_samples": 0.7},
]

OVERFIT_GAP_WEIGHT = 0.60

FS_MODE = "AUTO"
FS_RANK_METHOD = "TABNET"

FS_MAX_SAMPLES_TR  = 120000
FS_MAX_SAMPLES_VAL = 60000

FS_PERM_REPEATS = 3
FS_TOPK_CANDIDATES = [36, 30, 24, 18, 14, 12, 10, 8, 6]
FS_SIZE_WEIGHT = 0.01

FS_ALWAYS_KEEP = []

FS_TABNET_ENSEMBLE_RUNS  = 3
FS_TABNET_ENSEMBLE_SEEDS = [13, 37, 101]

TABNET_MAX_EPOCHS = 80
TABNET_PATIENCE   = 15
TABNET_BATCH_SIZE = 8192
TABNET_VBS        = 256
TABNET_LAMBDA_SPARSE = 1e-4
TABNET_MASK_TYPE = "sparsemax"

TABNET_N_D = 16
TABNET_N_A = 16
TABNET_N_STEPS = 5
TABNET_GAMMA = 1.5

FS_STABILITY_LAMBDA = 0.25

FS_REDUNDANCY_ALPHA = 0.35
FS_CORR_MAX_SAMPLES = 50000
FS_CORR_EPS = 1e-8

FS_USE_GROUP_MODE_WHEN_MULTIPLE_OF_3 = True
FS_GROUP_SCORE_AGG = "mean"

### 0.4 – 0.5. Debug, export coarse, cohérence multi-échelle

In [ ]:
# [SANS XAI] bloc 0.3 (paramètres XAI_* et INV_*) retiré

# =============================================================================
# 0.4) DEBUG / DIAGNOSTICS
# =============================================================================
DEBUG_ENABLE = True
DEBUG_TRIGGER_NVALID = 10
DEBUG_ALWAYS_FIRST_N_DATES = 2
DEBUG_SHOW_NAN_BY_FEATURE = True
DEBUG_DO_NOT_FORCE_4326_IN_LOAD = True

# =============================================================================
# 0.45) EXPORT COARSE (GLDAS resolution) AVANT SR ✅
# =============================================================================
EXPORT_COARSE_ENABLE = True

EXPORT_COARSE_TRAIN    = False   # souvent énorme -> False par défaut
EXPORT_COARSE_TEST     = True
EXPORT_COARSE_FORECAST = True

EXPORT_COARSE_USE_Y_MASK_FOR_TRAIN_TEST = True

COARSE_DIR = os.path.join(OUT_DIR, "COARSE_PRED")
os.makedirs(COARSE_DIR, exist_ok=True)

# =============================================================================
# 0.5) MULTI-SCALE CONSISTENCY (coarse mean conservation)
# =============================================================================
MS_ENABLE = True

# "UNIFORM" : offset constant par cellule coarse
# "WEIGHTED_A" : redistribution pondérée par A2d (hydro-cohérente)
MS_MODE = "WEIGHTED_A"

MS_EPS = 1e-6
MS_RATIO_CLIP_MIN = 0.2
MS_RATIO_CLIP_MAX = 5.0

# optionnel : exporter des diagnostics (delta coarse, etc.)
MS_EXPORT_DIAG = False
MS_DIAG_DIR = os.path.join(OUT_DIR, "MS_DIAG")
os.makedirs(MS_DIAG_DIR, exist_ok=True)

## 1. Utilitaires IO / dates + fusion

Adaptation Colab : les entrées de `name_files_SR.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# =============================================================================
# 1) UTILITAIRES IO / DATES + FUSION
# =============================================================================
def date_range(d0: date, d1: date):
    d = d0
    while d <= d1:
        yield d
        d += timedelta(days=1)

def parse_name_file(path_txt):
    sections = {}
    current = None
    with open(path_txt, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(line)
    return sections

def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p

def _norm_abs_from_entry(entry: str) -> str:
    entry = entry.strip().strip('"').strip("'")
    p = _win_to_colab_path(entry)  # [COLAB]
    if not os.path.isabs(p):
        p = os.path.join(BASE_DIR, p)
    return os.path.normpath(p)

def _scan_tifs_in_dir(d: str):
    out = []
    for ext in ("*.tif", "*.tiff", "*.TIF", "*.TIFF"):
        out.extend(glob.glob(os.path.join(d, "**", ext), recursive=True))
    return sorted(list(set(out)))

def _expand_entry_to_files(entry: str):
    p = _norm_abs_from_entry(entry)
    if any(ch in p for ch in ["*", "?", "["]):
        files = glob.glob(p, recursive=True)
        files = [os.path.normpath(x) for x in files if os.path.isfile(x)]
        return sorted(list(set(files)))
    if os.path.isdir(p):
        return _scan_tifs_in_dir(p)
    if os.path.isfile(p):
        return [p]
    return []

def _try_parse_yyyymmdd(s8: str):
    try:
        d = datetime.strptime(s8, "%Y%m%d").date()
        return d, s8
    except Exception:
        return None, None

def _try_parse_ddmmyyyy(s8: str):
    try:
        d = datetime.strptime(s8, "%d%m%Y").date()
        return d, d.strftime("%Y%m%d")
    except Exception:
        return None, None

def extract_date_yyyymmdd_from_any(path_str):
    b = os.path.basename(path_str)
    m = re.search(r"(\d{8})", b)
    if not m:
        return None
    s8 = m.group(1)

    d1, y1 = _try_parse_yyyymmdd(s8)
    if d1 is not None:
        return y1

    d2, y2 = _try_parse_ddmmyyyy(s8)
    if d2 is not None:
        return y2

    return None

def yyyymmdd_to_date(yyyymmdd: str) -> date:
    return datetime.strptime(yyyymmdd, "%Y%m%d").date()

def load_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
        profile["_bounds"] = tuple(src.bounds)
        profile["_crs_str"] = str(src.crs) if src.crs is not None else None

    if profile.get("crs") is None:
        if DEBUG_ENABLE:
            print(f"   [DEBUG] CRS manquant dans: {path}")
        if not (DEBUG_ENABLE and DEBUG_DO_NOT_FORCE_4326_IN_LOAD):
            profile["crs"] = CRS.from_epsg(4326)
    return data, profile, nodata

def to_float_with_nan(arr, nodata):
    a = arr.astype("float32")
    if nodata is not None:
        a = np.where(a == nodata, np.nan, a)
    return a

def reproject_to_profile(src_data, src_profile, src_nodata, dst_profile,
                         dst_nodata=np.nan, resampling=Resampling.bilinear):
    dst_data = np.full((dst_profile["height"], dst_profile["width"]), dst_nodata, dtype="float32")
    src_prof = src_profile.copy()
    if src_prof.get("crs") is None:
        src_prof["crs"] = dst_profile.get("crs", CRS.from_epsg(4326))

    reproject(
        source=src_data,
        destination=dst_data,
        src_transform=src_prof["transform"],
        src_crs=src_prof["crs"],
        src_nodata=src_nodata,
        dst_transform=dst_profile["transform"],
        dst_crs=dst_profile.get("crs", CRS.from_epsg(4326)),
        dst_nodata=dst_nodata,
        resampling=resampling
    )
    return dst_data

def write_geotiff(path, arr2d, profile, nodata_value=OUT_NODATA, compress="LZW"):
    p = {k: v for k, v in profile.items() if not str(k).startswith("_")}
    out_crs = p.get("crs", None) or CRS.from_epsg(4326)
    p.update(dtype="float32", count=1, nodata=nodata_value, compress=compress, crs=out_crs)

    out = arr2d.astype("float32")
    out = np.where(np.isfinite(out), out, nodata_value).astype("float32")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with rasterio.open(path, "w", **p) as dst:
        dst.write(out, 1)

def _norm_key(s: str) -> str:
    s = str(s or "")
    s = s.replace("\\", "/").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def _candidate_static_section_names(target_name: str):
    """
    Retourne une liste d'alias possibles pour trouver la section statique même si
    le .txt utilise un nom différent (ex: MNT vs DEM, Distance_To_River vs DTR).
    """
    base = str(target_name).strip()
    aliases = {
        "MNT": ["MNT", "DEM", "DTM", "MNE", "Elevation", "ModelNumeriqueTerrain", "Terrain"],
        "Distance_To_River": ["Distance_To_River", "DistanceToRiver", "DTR", "DistRiver", "RiverDistance"],
        "Slope": ["Slope", "Pente"],
        "TWI": ["TWI", "TopographicWetnessIndex", "WetnessIndex"],
        "SMAP_RZSM_Clipped": ["SMAP_RZSM_Clipped", "RZSM", "SMAP_RZSM"]
    }
    out = [base]
    out.extend(aliases.get(base, []))
    # uniques conservant l'ordre
    seen = set()
    final = []
    for x in out:
        k = _norm_key(x)
        if k not in seen:
            seen.add(k)
            final.append(x)
    return final

def _find_section_key_fuzzy(sections: dict, requested_key: str):
    """
    Cherche une clé de section dans 'sections' de façon tolérante (casse/underscore/alias).
    """
    if requested_key in sections:
        return requested_key

    # exact normalisé
    req_aliases = _candidate_static_section_names(requested_key)
    req_norms = {_norm_key(a) for a in req_aliases}

    for k in sections.keys():
        if _norm_key(k) in req_norms:
            return k

    # fuzzy léger: containment
    for k in sections.keys():
        kn = _norm_key(k)
        for rn in req_norms:
            if rn and (rn in kn or kn in rn):
                return k
    return None

def _guess_static_files_by_filename(base_dir, requested_label):
    """
    Fallback si la section n'existe pas (ou vide) dans name_files:
    recherche sur disque par nom de fichier probable.
    """
    candidates = []
    keys = _candidate_static_section_names(requested_label)
    keys_norm = [_norm_key(k) for k in keys]

    all_tifs = _scan_tifs_in_dir(base_dir)
    for fp in all_tifs:
        bn = _norm_key(os.path.basename(fp))
        dn = _norm_key(os.path.dirname(fp))
        fulln = _norm_key(fp)
        score = 0
        for k in keys_norm:
            if not k:
                continue
            if k in bn:
                score = max(score, 3)
            elif k in dn:
                score = max(score, 2)
            elif k in fulln:
                score = max(score, 1)
        # exclusions de sécurité pour éviter confusion dyn/stat
        if requested_label == "MNT":
            if any(x in bn for x in [_norm_key("chirps"), _norm_key("api"), _norm_key("gws"), _norm_key("rzsm")]):
                continue
        if score > 0:
            candidates.append((score, fp))

    candidates = sorted(candidates, key=lambda t: (-t[0], t[1]))
    return [fp for _, fp in candidates]

def export_csv_simple(path, header, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.write(",".join(header) + "\n")
        for r in rows:
            f.write(",".join(str(x) for x in r) + "\n")

def _safe_copy(src, dst, overwrite=False):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.isfile(dst) and (not overwrite):
        return False
    shutil.copy2(src, dst)
    return True

def fuse_sections_to_fused_dir(sections_raw: dict):
    """
    Fusionne les sections dyn + statiques en étant robuste aux noms de sections statiques
    manquants / alias dans name_files_SR.txt.
    """
    if not FUSION_ENABLE:
        return sections_raw, NAME_FILE

    print("\n================== FUSION INPUTS (anciens + nouveaux) ==================")
    print(" FUSION_ROOT:", FUSION_ROOT)
    print(" Policy:", FUSION_CONFLICT_POLICY, "| overwrite:", FUSION_OVERWRITE)
    os.makedirs(FUSION_ROOT, exist_ok=True)

    sections_fused = {}
    report_rows = []

    dyn_sections = {SECTION_CHIRPS, SECTION_API, SECTION_GWS, SECTION_RZSM}

    # 1) copier d'abord les sections déjà présentes (logique classique)
    for sec, entries in sections_raw.items():
        if sec not in FUSION_SECTIONS:
            sections_fused[sec] = entries[:]
            continue

        all_files = []
        for e in entries:
            files_e = _expand_entry_to_files(e)
            if not files_e:
                print(f"  ⚠️ [{sec}] entrée introuvable/ vide: {e}")
            all_files.append((e, files_e))

        is_dynamic = (sec in dyn_sections)
        dst_dir = os.path.join(FUSION_ROOT, sec)
        os.makedirs(dst_dir, exist_ok=True)

        if is_dynamic:
            chosen = {}  # YYYYMMDD -> (abs_path, entry_label)
            for entry_label, files_e in all_files:
                for fp in files_e:
                    dstr = extract_date_yyyymmdd_from_any(fp)
                    if dstr is None:
                        continue
                    if (dstr not in chosen) or (FUSION_CONFLICT_POLICY.upper() == "PREFER_LAST"):
                        chosen[dstr] = (fp, entry_label)

            rel_list = []
            for dstr in sorted(chosen.keys()):
                src_abs, entry_label = chosen[dstr]
                out_name = f"{sec}_{dstr}.tif"
                out_abs = os.path.join(dst_dir, out_name)
                copied = _safe_copy(src_abs, out_abs, overwrite=FUSION_OVERWRITE)
                out_rel = os.path.relpath(out_abs, BASE_DIR)
                rel_list.append(out_rel)
                report_rows.append([sec, dstr, entry_label, out_rel, os.path.basename(src_abs), int(copied)])

            sections_fused[sec] = rel_list
            print(f"  ✅ [{sec}] fusion dyn: {len(rel_list)} dates -> {os.path.relpath(dst_dir, BASE_DIR)}")
        else:
            cand = []
            for entry_label, files_e in all_files:
                for fp in files_e:
                    cand.append((entry_label, fp))

            if len(cand) == 0:
                sections_fused[sec] = []
                print(f"  ⚠️ [{sec}] aucun fichier statique trouvé (dans les entrées listées).")
                continue

            chosen_entry, chosen_fp = (cand[-1] if FUSION_CONFLICT_POLICY.upper() == "PREFER_LAST" else cand[0])
            out_name = f"{sec}.tif"
            out_abs = os.path.join(dst_dir, out_name)
            copied = _safe_copy(chosen_fp, out_abs, overwrite=FUSION_OVERWRITE)
            out_rel = os.path.relpath(out_abs, BASE_DIR)

            sections_fused[sec] = [out_rel]
            report_rows.append([sec, "STATIC", chosen_entry, out_rel, os.path.basename(chosen_fp), int(copied)])
            print(f"  ✅ [{sec}] fusion static: {out_rel}")

    # 2) GARANTIR la présence des sections statiques attendues (fallback alias + scan disque)
    expected_statics = [SECTION_SLOPE, SECTION_DTM, SECTION_TWI, SECTION_DTR]
    for target_sec in expected_statics:
        already_ok = (target_sec in sections_fused) and (len(sections_fused.get(target_sec, [])) > 0)
        if already_ok:
            continue

        # a) tenter alias de section dans sections_raw
        src_key = _find_section_key_fuzzy(sections_raw, target_sec)
        found_files = []
        source_label = None

        if src_key is not None:
            entries = sections_raw.get(src_key, [])
            for e in entries:
                found_files.extend(_expand_entry_to_files(e))
            found_files = sorted(list(set([p for p in found_files if os.path.isfile(p)])))
            if found_files:
                source_label = f"section_alias:{src_key}"

        # b) fallback scan disque (nom fichier / dossier)
        if not found_files:
            guessed = _guess_static_files_by_filename(BASE_DIR, target_sec)
            if guessed:
                found_files = guessed
                source_label = "disk_scan_fallback"

        if not found_files:
            sections_fused[target_sec] = []
            print(f"  ❌ [{target_sec}] introuvable (ni section, ni alias, ni scan disque).")
            continue

        chosen_fp = found_files[-1] if FUSION_CONFLICT_POLICY.upper() == "PREFER_LAST" else found_files[0]
        dst_dir = os.path.join(FUSION_ROOT, target_sec)
        os.makedirs(dst_dir, exist_ok=True)
        out_abs = os.path.join(dst_dir, f"{target_sec}.tif")
        copied = _safe_copy(chosen_fp, out_abs, overwrite=FUSION_OVERWRITE)
        out_rel = os.path.relpath(out_abs, BASE_DIR)

        sections_fused[target_sec] = [out_rel]
        report_rows.append([target_sec, "STATIC", source_label, out_rel, os.path.basename(chosen_fp), int(copied)])
        print(f"  ✅ [{target_sec}] fallback static: {out_rel} (source={source_label})")

    fused_name_file = os.path.join(FUSION_ROOT, "name_files_SR_FUSED.txt")
    with open(fused_name_file, "w", encoding="utf-8") as f:
        # Conserver ordre lisible: d'abord sections attendues si présentes, puis le reste
        preferred_order = [
            SECTION_CHIRPS, SECTION_API, SECTION_GWS, SECTION_RZSM,
            SECTION_SLOPE, SECTION_DTM, SECTION_TWI, SECTION_DTR
        ]
        done = set()

        for sec in preferred_order:
            if sec in sections_fused:
                f.write(f"[{sec}]\n")
                for r in sections_fused[sec]:
                    f.write(str(r).replace("\\", "/") + "\n")
                f.write("\n")
                done.add(sec)

        for sec, rels in sections_fused.items():
            if sec in done:
                continue
            f.write(f"[{sec}]\n")
            for r in rels:
                f.write(str(r).replace("\\", "/") + "\n")
            f.write("\n")

    export_csv_simple(
        os.path.join(FUSION_ROOT, "fusion_report.csv"),
        ["section", "date_or_static", "source_entry", "fused_rel_path", "source_basename", "copied(1/0)"],
        report_rows
    )
    print(" ✅ name_files_SR_FUSED.txt généré + fusion_report.csv")
    print("=======================================================================\n")
    return sections_fused, fused_name_file


def _subsample_xy(X, y, max_n, seed=42):
    n = X.shape[0]
    if (max_n is None) or (n <= max_n):
        return X, y
    rng = np.random.RandomState(seed)
    idx = rng.choice(n, size=max_n, replace=False)
    return X[idx], y[idx]

def _subsample_X(X, max_n, seed=42):
    n = X.shape[0]
    if (max_n is None) or (n <= max_n):
        return X
    rng = np.random.RandomState(seed)
    idx = rng.choice(n, size=max_n, replace=False)
    return X[idx]

def _nan_stats(arr2d):
    a = arr2d.astype("float32")
    n = a.size
    nn = int(np.count_nonzero(np.isnan(a)))
    nf = int(np.count_nonzero(np.isfinite(a)))
    nan_pct = 100.0 * nn / max(1, n)
    fin_pct = 100.0 * nf / max(1, n)
    mn = float(np.nanmin(a)) if nf > 0 else np.nan
    mx = float(np.nanmax(a)) if nf > 0 else np.nan
    return nan_pct, fin_pct, mn, mx

def _print_profile(tag, prof):
    crs = prof.get("crs", None)
    tr = prof.get("transform", None)
    b = prof.get("_bounds", None)
    print(f"   [DEBUG] {tag} crs={crs} bounds={b} transform={tr}")

def debug_report_date(tag_date, fine_prof, coarse_prof, y_coarse_2d, dyn_fine, stat_fine, feat_names, Xc_mask_valid_2d, extra_msg=""):
    print("\n" + "=" * 88)
    print(f"[DEBUG] DATE = {tag_date} {extra_msg}")
    print("=" * 88)
    _print_profile("FINE(template=CHIRPS)", fine_prof)
    _print_profile("COARSE(template=GWS)", coarse_prof)
    if y_coarse_2d is not None:
        nan_pct, fin_pct, mn, mx = _nan_stats(y_coarse_2d)
        print(f"   [DEBUG] y(GWS coarse) nan%={nan_pct:6.2f} fin%={fin_pct:6.2f} min={mn} max={mx}")
    if dyn_fine:
        print("   [DEBUG] --- DYN (fine) ---")
        for k, a2d in dyn_fine.items():
            nan_pct, fin_pct, mn, mx = _nan_stats(a2d)
            print(f"   [DEBUG] {k:14s} nan%={nan_pct:6.2f} fin%={fin_pct:6.2f} min={mn} max={mx}")
    if stat_fine:
        print("   [DEBUG] --- STATIC (fine) ---")
        for k, a2d in stat_fine.items():
            nan_pct, fin_pct, mn, mx = _nan_stats(a2d)
            print(f"   [DEBUG] {k:14s} nan%={nan_pct:6.2f} fin%={fin_pct:6.2f} min={mn} max={mx}")
    if Xc_mask_valid_2d is not None:
        nvalid = int(np.count_nonzero(Xc_mask_valid_2d))
        print(f"   [DEBUG] mask_valid(coarse) nvalid={nvalid} / {Xc_mask_valid_2d.size}")
    if DEBUG_SHOW_NAN_BY_FEATURE and feat_names:
        print("   [DEBUG] features:", len(feat_names), "ex:", feat_names[:min(12, len(feat_names))], "...")

### Contrôle des entrées de `name_files_SR.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichiers présents sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files_SR.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _n_files, _missing = 0, []
    for _e in _entries:
        _f = _expand_entry_to_files(_e)
        _n_files += len(_f)
        if not _f:
            _missing.append(_e)
    _flag = "✅" if (_n_files > 0 and not _missing) else ("⚠️" if _n_files > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_files} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {_e}  ->  {_norm_abs_from_entry(_e)}")

## 2. HADA + helpers d'attention

In [ ]:
# =============================================================================
# 2) HADA + ATTENTION HELPERS
# =============================================================================
def _zscore(a2d: np.ndarray, eps=1e-6) -> np.ndarray:
    a = a2d.astype(np.float32)
    m = float(np.nanmean(a)) if np.any(np.isfinite(a)) else 0.0
    s = float(np.nanstd(a)) if np.any(np.isfinite(a)) else 1.0
    if s < eps:
        return np.zeros_like(a, dtype=np.float32)
    z = (a - m) / (s + eps)
    z = np.where(np.isfinite(z), z, 0.0).astype(np.float32)
    return z

def _grad_mag_fast(a2d: np.ndarray) -> np.ndarray:
    a = a2d.astype(np.float32)
    a = np.where(np.isfinite(a), a, 0.0).astype(np.float32)
    dx = np.abs(np.diff(a, axis=1, prepend=a[:, :1]))
    dy = np.abs(np.diff(a, axis=0, prepend=a[:1, :]))
    return (dx + dy).astype(np.float32)

def sigmoid(x):
    x = np.clip(x, -30.0, 30.0).astype(np.float32)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

def build_hada_attention(twi2d, dtr2d, api2d, chirps2d, slope2d):
    dtr = dtr2d.astype(np.float32)
    dtrv = dtr[np.isfinite(dtr)]
    if dtrv.size > 0:
        q90 = float(np.quantile(dtrv, 0.90))
        q90 = max(q90, 1e-6)
    else:
        q90 = 1.0
    dtr_n = np.clip(dtr / q90, 0.0, 5.0).astype(np.float32)
    corridor = np.exp(- (dtr_n / float(HADA_LAM)) ** 2).astype(np.float32)

    z_twi   = _zscore(twi2d)
    z_api   = _zscore(api2d)
    z_ch    = _zscore(chirps2d)
    z_slp   = _zscore(slope2d)

    logits = (HADA_A0
              + HADA_A1 * z_twi
              + HADA_A2 * corridor
              + HADA_A3 * z_api
              + HADA_A4 * z_ch
              - HADA_A5 * z_slp)
    A = sigmoid(logits)
    return A.astype(np.float32)

def build_attention_map_simple(src_planes, use_grad=True):
    score = None
    for a in src_planes:
        s = _zscore(a)
        if use_grad:
            s = s + float(FE_ATT_GRAD_W) * _zscore(_grad_mag_fast(a))
        score = s if score is None else (score + s)
    return sigmoid(float(FE_ATT_BETA) * score)

## 3. Graphe hydrologique D8 + Hydro-Attention (robuste aux NaN)

In [ ]:
# =============================================================================
# 3) HYDROGRAPH D8 + HYDRO-ATT (robuste aux NaN)
# =============================================================================
HYDRO_GRAPH = None

def _shift_with_inf(dem2d: np.ndarray, dy: int, dx: int) -> np.ndarray:
    dem = dem2d.astype(np.float32)
    out = np.roll(np.roll(dem, -dy, axis=0), -dx, axis=1).astype(np.float32)
    if dy > 0:
        out[-dy:, :] = np.inf
    elif dy < 0:
        out[:(-dy), :] = np.inf
    if dx > 0:
        out[:, -dx:] = np.inf
    elif dx < 0:
        out[:, :(-dx)] = np.inf
    return out

def build_hydro_graph_from_dem(dem2d: np.ndarray):
    dem = dem2d.astype(np.float32)
    dem = np.where(np.isfinite(dem), dem, np.inf).astype(np.float32)

    H, W = dem.shape
    N = H * W

    dirs = [(-1, -1), (-1, 0), (-1, 1),
            ( 0, -1),          ( 0, 1),
            ( 1, -1), ( 1, 0), ( 1, 1)]

    delta_stack = []
    for dy, dx in dirs:
        neigh = _shift_with_inf(dem, dy, dx)
        delta = neigh - dem
        delta = np.where(delta < 0.0, delta, np.inf).astype(np.float32)
        delta_stack.append(delta)

    delta_stack = np.stack(delta_stack, axis=0)
    best_dir = np.argmin(delta_stack, axis=0).astype(np.int32)
    best_delta = np.min(delta_stack, axis=0).astype(np.float32)

    rows = np.arange(H, dtype=np.int32)[:, None]
    cols = np.arange(W, dtype=np.int32)[None, :]
    ds_r = rows + np.array([d[0] for d in dirs], dtype=np.int32)[best_dir]
    ds_c = cols + np.array([d[1] for d in dirs], dtype=np.int32)[best_dir]

    has_ds = np.isfinite(best_delta)
    ds_r = np.where(has_ds, ds_r, -1)
    ds_c = np.where(has_ds, ds_c, -1)

    downstream = np.full((N,), -1, dtype=np.int32)
    flat_i = (rows * W + cols).ravel()
    flat_ds = (ds_r * W + ds_c).ravel()

    m = has_ds.ravel()
    downstream[flat_i[m]] = flat_ds[m].astype(np.int32)

    upstream_idx = np.full((N, 8), -1, dtype=np.int32)
    upstream_mask = (upstream_idx >= 0)
    upstream_count = np.zeros((N,), dtype=np.int32)

    src = np.nonzero(downstream >= 0)[0].astype(np.int32)
    dst = downstream[src].astype(np.int32)

    order = np.argsort(dst, kind="mergesort")
    dst_s = dst[order]
    src_s = src[order]

    unique_dst, start_idx, counts = np.unique(dst_s, return_index=True, return_counts=True)
    for d, st, cnt in zip(unique_dst.tolist(), start_idx.tolist(), counts.tolist()):
        k = min(int(cnt), 8)
        upstream_idx[d, :k] = src_s[st:st+k]
        upstream_count[d] = k

    upstream_mask = (upstream_idx >= 0)

    dem_flat = dem.ravel().astype(np.float32)
    dem_fin = dem_flat[np.isfinite(dem_flat)]
    dem_std = float(np.std(dem_fin)) if dem_fin.size > 0 else 1.0
    if dem_std < 1e-6:
        dem_std = 1.0

    return {
        "H": H, "W": W, "N": N,
        "downstream": downstream,
        "upstream_idx": upstream_idx,
        "upstream_mask": upstream_mask,
        "dem_flat": dem_flat,
        "dem_std": dem_std,
    }

def hydro_att_softmax_weights(A_flat, dem_flat, dem_std,
                             upstream_idx, upstream_mask,
                             beta_att, dz_w, eps=1e-8):
    N, K = upstream_idx.shape
    msk = upstream_mask

    idx_eff = np.where(msk, upstream_idx, 0).astype(np.int32)

    A_nbr = A_flat[idx_eff]
    dem_nbr = dem_flat[idx_eff]
    dem_i = dem_flat.reshape(-1, 1)

    dz = dem_nbr - dem_i
    dz_pos = np.maximum(dz, 0.0) / (dem_std + eps)

    logits = beta_att * A_nbr + dz_w * dz_pos
    very_neg = -1e9
    logits = np.where(msk, logits, very_neg).astype(np.float32)

    max_l = np.max(logits, axis=1, keepdims=True)
    exp_l = np.exp(logits - max_l).astype(np.float32)
    exp_l = np.where(msk, exp_l, 0.0).astype(np.float32)
    sum_exp = np.sum(exp_l, axis=1, keepdims=True).astype(np.float32)

    w = exp_l / (sum_exp + eps)

    no_nbr = (sum_exp[:, 0] <= eps)
    if np.any(no_nbr):
        idx_self = np.arange(N, dtype=np.int32)
        idx_eff[no_nbr, :] = 0
        w[no_nbr, :] = 0.0
        idx_eff[no_nbr, 0] = idx_self[no_nbr]
        w[no_nbr, 0] = 1.0

    return idx_eff, w.astype(np.float32)

def hydro_att_agg_upstream(x_flat, idx_eff, w):
    x_flat = x_flat.astype(np.float32)
    x_nbr = x_flat[idx_eff]
    y = np.sum(w * x_nbr, axis=1).astype(np.float32)
    return y

def hydro_mean_std_hops(a2d, A2d, graph, hops):
    H, W = graph["H"], graph["W"]
    upstream_idx = graph["upstream_idx"]
    upstream_mask = graph["upstream_mask"]
    dem_flat = graph["dem_flat"]
    dem_std = graph["dem_std"]

    x = np.where(np.isfinite(a2d), a2d.astype(np.float32), 0.0).reshape(-1).astype(np.float32)
    x2 = (x * x).astype(np.float32)
    A_flat = np.where(np.isfinite(A2d), A2d.astype(np.float32), 0.0).reshape(-1).astype(np.float32)

    for s in range(max(1, int(hops))):
        scale = float(FE_HOP_DECAY ** s)
        idx_eff, w = hydro_att_softmax_weights(
            A_flat=A_flat,
            dem_flat=dem_flat,
            dem_std=dem_std,
            upstream_idx=upstream_idx,
            upstream_mask=upstream_mask,
            beta_att=float(FE_BETA_ATT) * scale,
            dz_w=float(FE_DZ_W) * scale,
            eps=FE_SOFT_EPS
        )
        x  = hydro_att_agg_upstream(x,  idx_eff, w)
        x2 = hydro_att_agg_upstream(x2, idx_eff, w)

    var = np.maximum(x2 - x * x, 0.0).astype(np.float32)
    std = np.sqrt(var).astype(np.float32)
    return x.reshape(H, W).astype(np.float32), std.reshape(H, W).astype(np.float32)

def build_feature_stack_hydro(raw_planes_2d, raw_names, enable, A2d, graph, mask_valid_fine):
    feat_planes = [rp.astype(np.float32) for rp in raw_planes_2d]
    feat_names = list(raw_names)

    if not enable:
        return feat_planes, feat_names

    for a, n in zip(raw_planes_2d, raw_names):
        m2d, s2d = hydro_mean_std_hops(a.astype(np.float32), A2d.astype(np.float32), graph=graph, hops=FE_HOPS)

        if mask_valid_fine is not None:
            m2d = np.where(mask_valid_fine, m2d, np.nan).astype(np.float32)
            s2d = np.where(mask_valid_fine, s2d, np.nan).astype(np.float32)

        feat_planes.append(m2d)
        feat_names.append(f"{n}_hydro_wmean_h{FE_HOPS}")

        feat_planes.append(s2d)
        feat_names.append(f"{n}_hydro_wstd_h{FE_HOPS}")

    return feat_planes, feat_names

## 4. Helpers Random Forest / sélection de variables

In [ ]:
# =============================================================================
# 4) RF/FS HELPERS
# =============================================================================
def _supported_params_only(param_dict):
    probe = RandomForestRegressor()
    allowed = set(probe.get_params().keys())
    return {k: v for k, v in param_dict.items() if k in allowed}

def _fit_and_score_rf(X_tr, y_tr, X_val, y_val, params, seed=42, sample_weight_tr=None):
    full_params = {}
    full_params.update(RF_BASE_PARAMS)
    full_params.update(params)
    full_params = _supported_params_only(full_params)

    model = RandomForestRegressor(
        **full_params,
        n_jobs=-1,
        random_state=seed,
        oob_score=True,
    )

    t0 = time.time()
    try:
        model.fit(X_tr, y_tr, sample_weight=sample_weight_tr)
    except TypeError:
        model.fit(X_tr, y_tr)
    t1 = time.time()

    ytr_pred = model.predict(X_tr)
    yval_pred = model.predict(X_val)

    r2_tr = r2_score(y_tr, ytr_pred)
    r2_val = r2_score(y_val, yval_pred)
    gap = max(0.0, r2_tr - r2_val)

    oob_r2 = getattr(model, "oob_score_", np.nan)
    sel_score = r2_val - OVERFIT_GAP_WEIGHT * gap

    return {
        "model": model,
        "fit_ms": (t1 - t0) * 1000.0,
        "r2_tr": r2_tr,
        "r2_val": r2_val,
        "gap": gap,
        "oob_r2": oob_r2,
        "sel_score": sel_score,
        "params_used": full_params,
    }

def choose_regularized_params(X_tr, y_tr, X_val, y_val, w_tr=None):
    print("\n=== SÉLECTION AUTO DES PARAMÈTRES RF RÉGULARISÉS (ANTI-OVERFITTING) ===")
    best = None
    for i, cand in enumerate(RF_REG_CANDIDATES, start=1):
        info = _fit_and_score_rf(X_tr, y_tr, X_val, y_val, cand, seed=42, sample_weight_tr=w_tr)
        print(f"  Candidate #{i}: R²_val={info['r2_val']:.4f} gap={info['gap']:.4f} sel={info['sel_score']:.4f}")
        if (best is None) or (info["sel_score"] > best["sel_score"]):
            best = info
    kept = {k: best["params_used"][k] for k in best["params_used"] if k not in ["n_jobs", "random_state", "oob_score"]}
    print("✅ Config retenue :", kept)
    return best["params_used"]

def build_grouped_features_for_fe(n_raw: int, fe_enabled: bool):
    if not fe_enabled:
        return None
    groups = []
    for i in range(n_raw):
        raw_idx = i
        mean_idx = n_raw + 2 * i
        std_idx  = n_raw + 2 * i + 1
        groups.append([raw_idx, mean_idx, std_idx])
    return groups

def _compute_abs_corr_matrix(X: np.ndarray, eps=1e-8) -> np.ndarray:
    n, p = X.shape
    if n < 2:
        return np.zeros((p, p), dtype=np.float32)

    X = X.astype(np.float32)
    mu = np.mean(X, axis=0, keepdims=True)
    Xc = X - mu
    sd = np.std(Xc, axis=0, keepdims=True)
    sd_safe = np.where(sd < eps, 1.0, sd)
    Xn = Xc / sd_safe

    corr = (Xn.T @ Xn) / float(max(1, n - 1))
    corr = np.clip(corr, -0.999, 0.999).astype(np.float32)

    const = (sd.reshape(-1) < eps)
    if np.any(const):
        corr[const, :] = 0.0
        corr[:, const] = 0.0

    np.fill_diagonal(corr, 1.0)
    return np.abs(corr).astype(np.float32)

def _rank_features_perm_stable(model, X_val, y_val):
    if not _HAS_PERM_IMPORTANCE:
        if hasattr(model, "feature_importances_"):
            imps = np.asarray(model.feature_importances_, dtype=np.float64)
            return imps, np.zeros_like(imps)
        return np.ones(X_val.shape[1], dtype=np.float64), np.zeros(X_val.shape[1], dtype=np.float64)

    Xv, yv = _subsample_xy(X_val, y_val, FS_MAX_SAMPLES_VAL, seed=42)
    runs = 3
    seeds = [42, 7, 999]
    all_imps = []

    for s in seeds[:runs]:
        perm = permutation_importance(
            model, Xv, yv,
            n_repeats=FS_PERM_REPEATS,
            random_state=s,
            scoring="r2",
            n_jobs=-1
        )
        all_imps.append(np.asarray(perm.importances_mean, dtype=np.float64))

    M = np.stack(all_imps, axis=0)
    return np.mean(M, axis=0), np.std(M, axis=0)

def _rank_features_tabnet_ensemble(X_tr, y_tr, X_val, y_val, n_raw, fe_enabled):
    if not _HAS_TABNET:
        return None, None

    Xt, yt = _subsample_xy(X_tr, y_tr, FS_MAX_SAMPLES_TR, seed=777)
    Xv, yv = _subsample_xy(X_val, y_val, FS_MAX_SAMPLES_VAL, seed=42)

    Xt = Xt.astype("float32")
    Xv = Xv.astype("float32")
    yt2 = yt.astype("float32").reshape(-1, 1)
    yv2 = yv.astype("float32").reshape(-1, 1)

    grouped = build_grouped_features_for_fe(n_raw, fe_enabled)

    all_imps = []
    for run_id, seed in enumerate(FS_TABNET_ENSEMBLE_SEEDS[:FS_TABNET_ENSEMBLE_RUNS], start=1):
        print(f"🧠 TabNet run {run_id}/{FS_TABNET_ENSEMBLE_RUNS} (seed={seed}) ...")
        tabnet = TabNetRegressor(
            n_d=TABNET_N_D,
            n_a=TABNET_N_A,
            n_steps=TABNET_N_STEPS,
            gamma=TABNET_GAMMA,
            lambda_sparse=TABNET_LAMBDA_SPARSE,
            mask_type=TABNET_MASK_TYPE,
            grouped_features=grouped,
            seed=seed,
            verbose=0,
            device_name="auto",
        )
        tabnet.fit(
            Xt, yt2,
            eval_set=[(Xv, yv2)],
            eval_metric=["rmse"],
            max_epochs=TABNET_MAX_EPOCHS,
            patience=TABNET_PATIENCE,
            batch_size=TABNET_BATCH_SIZE,
            virtual_batch_size=TABNET_VBS,
            num_workers=0,
            drop_last=False,
        )
        imps = np.asarray(getattr(tabnet, "feature_importances_", None), dtype=np.float64)
        if (imps is None) or (imps.shape[0] != X_tr.shape[1]):
            return None, None
        all_imps.append(imps)

    M = np.stack(all_imps, axis=0)
    return np.mean(M, axis=0), np.std(M, axis=0)

def _make_ranked_list(imp_mean, imp_std, stability_lambda):
    score = imp_mean - stability_lambda * imp_std
    order = np.argsort(-score)
    return order, score

def _build_selected_indices_mrmr(order, score_vec, abs_corr, top_k, always_keep_idx):
    selected = set(always_keep_idx)
    target = max(int(top_k), len(selected))
    candidates = [int(i) for i in order.tolist() if int(i) not in selected]

    while (len(selected) < target) and candidates:
        best_idx = None
        best_obj = -1e18
        sel_list = list(selected)
        for cand in candidates:
            base = float(score_vec[cand])
            red = 0.0 if len(sel_list) == 0 else float(np.mean(abs_corr[cand, sel_list]))
            obj = base - FS_REDUNDANCY_ALPHA * red
            if obj > best_obj:
                best_obj = obj
                best_idx = cand
        if best_idx is None:
            break
        selected.add(int(best_idx))
        candidates.remove(best_idx)

    return sorted(list(selected))

def _build_selected_indices_group_mode(score_vec, abs_corr, top_k, always_keep_idx, n_raw):
    groups = build_grouped_features_for_fe(n_raw, fe_enabled=True)
    if not groups:
        return list(range(len(score_vec)))

    n_groups_target = max(1, int(top_k // 3))
    gscore = np.zeros((len(groups),), dtype=np.float64)
    for gi, members in enumerate(groups):
        vals = np.asarray([score_vec[m] for m in members], dtype=np.float64)
        gscore[gi] = float(np.mean(vals)) if FS_GROUP_SCORE_AGG.lower() != "sum" else float(np.sum(vals))

    selected_groups = set()
    for gi, members in enumerate(groups):
        if any(m in always_keep_idx for m in members):
            selected_groups.add(gi)

    candidates = [gi for gi in np.argsort(-gscore).tolist() if gi not in selected_groups]

    while (len(selected_groups) < n_groups_target) and candidates:
        best_gi = None
        best_obj = -1e18
        sel_g_list = list(selected_groups)
        for gi in candidates:
            base = float(gscore[gi])
            if len(sel_g_list) == 0:
                red = 0.0
            else:
                red_vals = []
                for sj in sel_g_list:
                    mm = groups[gi]
                    ss = groups[sj]
                    sub = abs_corr[np.ix_(mm, ss)]
                    red_vals.append(float(np.max(sub)))
                red = float(np.mean(red_vals)) if red_vals else 0.0
            obj = base - FS_REDUNDANCY_ALPHA * red
            if obj > best_obj:
                best_obj = obj
                best_gi = gi
        if best_gi is None:
            break
        selected_groups.add(int(best_gi))
        candidates.remove(best_gi)

    selected = set()
    for gi in selected_groups:
        for m in groups[gi]:
            selected.add(int(m))
    for i in always_keep_idx:
        selected.add(int(i))
    return sorted(list(selected))

def _score_subset_with_penalty(rf_params, X_tr, y_tr, X_val, y_val, w_tr, feat_idx, total_feats):
    Xt = X_tr[:, feat_idx]
    Xv = X_val[:, feat_idx]

    # subsample train
    n = Xt.shape[0]
    if (FS_MAX_SAMPLES_TR is not None) and (n > FS_MAX_SAMPLES_TR):
        rng = np.random.RandomState(123)
        idx = rng.choice(n, size=FS_MAX_SAMPLES_TR, replace=False)
        Xt_s = Xt[idx]
        yt_s = y_tr[idx]
        wts = (w_tr[idx] if w_tr is not None else None)
    else:
        Xt_s = Xt
        yt_s = y_tr
        wts = (w_tr if w_tr is not None else None)

    rf = RandomForestRegressor(
        **rf_params,
        n_jobs=-1,
        random_state=42,
        oob_score=True,
    )
    try:
        rf.fit(Xt_s, yt_s, sample_weight=wts)
    except TypeError:
        rf.fit(Xt_s, yt_s)

    ytr_pred = rf.predict(Xt_s)
    yval_pred = rf.predict(Xv)

    r2_tr = r2_score(yt_s, ytr_pred)
    r2_val = r2_score(y_val, yval_pred)
    gap = max(0.0, r2_tr - r2_val)

    k = len(feat_idx)
    size_penalty = FS_SIZE_WEIGHT * (k / float(total_feats))
    sel_score = r2_val - OVERFIT_GAP_WEIGHT * gap - size_penalty

    return {"r2_tr": r2_tr, "r2_val": r2_val, "gap": gap, "k": k, "sel_score": sel_score, "feat_idx": feat_idx}

def run_feature_selection(rf_params_used, X_tr, y_tr, X_val, y_val, feature_names, n_raw, w_tr=None):
    print("\n=== FEATURE SELECTION (AUTO) ✅ stable ranking + mRMR + group-hydro ===")
    total_feats = len(feature_names)

    Xc = _subsample_X(X_val, FS_CORR_MAX_SAMPLES, seed=202)
    abs_corr = _compute_abs_corr_matrix(Xc, eps=FS_CORR_EPS)

    imp_mean = None
    imp_std  = None

    if FS_RANK_METHOD.upper() == "TABNET":
        imp_mean, imp_std = _rank_features_tabnet_ensemble(
            X_tr, y_tr, X_val, y_val,
            n_raw=n_raw,
            fe_enabled=FE_ENABLE
        )

    if imp_mean is None:
        print("↪️ Fallback ranking : PERM/RF (stable)")
        base_rf = RandomForestRegressor(**rf_params_used, n_jobs=-1, random_state=42, oob_score=True)
        Xt_s, yt_s = _subsample_xy(X_tr, y_tr, FS_MAX_SAMPLES_TR, seed=777)
        try:
            sw = None
            if w_tr is not None:
                if Xt_s.shape[0] == X_tr.shape[0]:
                    sw = w_tr
                else:
                    # fallback sécurisé (indices inconnus après subsample_xy)
                    sw = None
            base_rf.fit(Xt_s, yt_s, sample_weight=sw)
        except Exception:
            try:
                base_rf.fit(X_tr, y_tr, sample_weight=w_tr)
            except Exception:
                base_rf.fit(X_tr, y_tr)
        imp_mean, imp_std = _rank_features_perm_stable(base_rf, X_val, y_val)

    order, score_vec = _make_ranked_list(imp_mean, imp_std, stability_lambda=FS_STABILITY_LAMBDA)

    export_csv_simple(
        os.path.join(OUT_DIR, "feature_ranking_stable.csv"),
        ["idx", "feature", "score_stable", "importance_mean", "importance_std"],
        [(int(i), feature_names[int(i)], float(score_vec[int(i)]), float(imp_mean[int(i)]), float(imp_std[int(i)]))
         for i in order.tolist()]
    )

    cand_topk = list(FS_TOPK_CANDIDATES)
    if total_feats not in cand_topk:
        cand_topk.append(total_feats)
    cand_topk = sorted(list(set([k for k in cand_topk if 1 <= k <= total_feats])), reverse=True)

    always_keep_idx = []
    name_to_idx = {n: i for i, n in enumerate(feature_names)}
    for n in FS_ALWAYS_KEEP:
        if n in name_to_idx:
            always_keep_idx.append(int(name_to_idx[n]))

    best = None
    for k in cand_topk:
        use_group = (
            FE_ENABLE
            and FS_USE_GROUP_MODE_WHEN_MULTIPLE_OF_3
            and (k % 3 == 0)
            and (total_feats == 3 * n_raw)
        )

        if use_group:
            feat_idx = _build_selected_indices_group_mode(score_vec, abs_corr, k, always_keep_idx, n_raw=n_raw)
        else:
            feat_idx = _build_selected_indices_mrmr(order, score_vec, abs_corr, k, always_keep_idx)

        info = _score_subset_with_penalty(rf_params_used, X_tr, y_tr, X_val, y_val, w_tr, feat_idx, total_feats)
        if (best is None) or (info["sel_score"] > best["sel_score"]):
            best = info

    best_names = [feature_names[i] for i in best["feat_idx"]]
    print("✅ FS retenu : |S|=", len(best["feat_idx"]), "R²_val=", f"{best['r2_val']:.4f}", "gap=", f"{best['gap']:.4f}")
    return best["feat_idx"], best_names

## 5. Prédiction par blocs

In [ ]:
# =============================================================================
# 5) CHUNKED PREDICTION
# =============================================================================
# [SANS XAI] rf_predict_std_chunked retiré (ne servait qu'aux cartes d'incertitude XAI)

def predict_chunked(model, X, chunk=200000):
    n = X.shape[0]
    out = np.empty((n,), dtype=np.float32)
    for i0 in range(0, n, chunk):
        i1 = min(n, i0 + chunk)
        out[i0:i1] = model.predict(X[i0:i1]).astype(np.float32)
    return out

## 7. Construction du dataset + mapping SR

In [ ]:
# =============================================================================
# 7) CONSTRUCTION DATASET + MAPPING SR
# =============================================================================
REGION_MASK_FINE = None

def update_region_mask_fine(*dyn_fine):
    global REGION_MASK_FINE
    base = None
    for a in dyn_fine:
        if a is None:
            continue
        m = np.isfinite(a)
        base = m if base is None else (base | m)
    if base is None:
        return
    if REGION_MASK_FINE is None:
        REGION_MASK_FINE = base
        print("\n🗺️ Masque 'Regions' (fine) initial construit (1er jour).")
    else:
        REGION_MASK_FINE = REGION_MASK_FINE | base

def get_region_mask_fine(h, w):
    global REGION_MASK_FINE
    if REGION_MASK_FINE is None:
        print("⚠️ REGION_MASK_FINE non initialisé -> toute la grille fine.")
        REGION_MASK_FINE = np.ones((h, w), dtype=bool)
    return REGION_MASK_FINE

STATIC_FINE_CACHE = {}

def ensure_static_on_fine(static_path, fine_profile, label):
    key = (label, fine_profile["height"], fine_profile["width"], str(fine_profile.get("crs")))
    if key in STATIC_FINE_CACHE:
        return STATIC_FINE_CACHE[key]

    a, p, n = load_raster(static_path)
    a = to_float_with_nan(a, n)
    a_f = reproject_to_profile(a, p, n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)
    if np.any(np.isfinite(a_f)):
        mu = float(np.nanmean(a_f))
        a_f = np.where(np.isfinite(a_f), a_f, mu).astype(np.float32)
    else:
        a_f = np.zeros_like(a_f, dtype=np.float32)

    STATIC_FINE_CACHE[key] = a_f.astype(np.float32)
    print(f"✅ {label} reprojeté sur grille FINE.")
    return STATIC_FINE_CACHE[key]

def ensure_hydro_graph_fine(dem_fine, fine_profile):
    global HYDRO_GRAPH
    if HYDRO_GRAPH is not None:
        return HYDRO_GRAPH
    t0 = time.time()
    HYDRO_GRAPH = build_hydro_graph_from_dem(dem_fine)
    t1 = time.time()
    H = HYDRO_GRAPH["H"]; W = HYDRO_GRAPH["W"]; N = HYDRO_GRAPH["N"]
    sinks = int(np.sum(HYDRO_GRAPH["downstream"] < 0))
    print(f"\n🌊 Graphe D8 (FINE) construit H={H}, W={W}, N={N}, sinks={sinks} en {(t1-t0)*1000:.2f} ms")
    return HYDRO_GRAPH

def build_date_to_abs_path_dict(sections, section_name):
    """
    Construit dict date->path avec résolution tolérante du nom de section.
    """
    sec_key = _find_section_key_fuzzy(sections, section_name)
    if sec_key is None:
        return {}
    d = {}
    for rel_path in sections.get(sec_key, []):
        abs_path = _norm_abs_from_entry(rel_path)  # [COLAB] tolère les chemins Windows
        if not os.path.isfile(abs_path):
            print(f"⚠️ introuvable: {abs_path}")
            continue
        yyyymmdd = extract_date_yyyymmdd_from_any(abs_path)
        if yyyymmdd is None:
            continue
        d[yyyymmdd_to_date(yyyymmdd)] = abs_path
    return d

def get_single_static_path(sections, section_name):
    """
    Récupère le path d'un raster statique de façon robuste:
    1) section exacte
    2) section alias/fuzzy
    3) si l'entrée est vide -> scan disque fallback
    """
    sec_key = _find_section_key_fuzzy(sections, section_name)
    if sec_key is not None and len(sections.get(sec_key, [])) > 0:
        rel = sections[sec_key][0]
        p = _norm_abs_from_entry(rel)  # [COLAB] tolère les chemins Windows
        if os.path.isfile(p):
            return p

    # fallback scan disque
    guessed = _guess_static_files_by_filename(BASE_DIR, section_name)
    for p in guessed:
        if os.path.isfile(p):
            print(f"⚠️ Fallback static [{section_name}] via scan disque: {p}")
            return p
    return None

def build_coarse_features_from_fine_planes(fine_planes, fine_profile, coarse_profile):
    coarse_planes = []
    for fp in fine_planes:
        src = np.where(np.isfinite(fp), fp, 0.0).astype(np.float32)
        cp = reproject_to_profile(src, fine_profile, src_nodata=None, dst_profile=coarse_profile,
                                  dst_nodata=np.nan, resampling=Resampling.average)
        coarse_planes.append(cp.astype(np.float32))
    return coarse_planes

def maybe_add_sin_cos(raw_planes, raw_names, fine_profile, d: date):
    if not USE_SIN_COS_DOY:
        return raw_planes, raw_names
    doy = d.timetuple().tm_yday
    sin_doy = float(np.sin(2 * np.pi * doy / 365.0))
    cos_doy = float(np.cos(2 * np.pi * doy / 365.0))
    H = fine_profile["height"]; W = fine_profile["width"]
    raw_planes.append(np.full((H, W), sin_doy, dtype=np.float32))
    raw_names.append("sinDOY")
    raw_planes.append(np.full((H, W), cos_doy, dtype=np.float32))
    raw_names.append("cosDOY")
    return raw_planes, raw_names

def enforce_multiscale_consistency(
    pred_fine_2d: np.ndarray,
    fine_profile: dict,
    coarse_target_2d: np.ndarray,
    coarse_profile: dict,
    mask_pred_fine: np.ndarray = None,
    A2d: np.ndarray = None,
    mode: str = "UNIFORM",
    eps: float = 1e-6,
    ratio_clip=(0.2, 5.0),
):
    """
    Ajuste pred_fine_2d pour que sa moyenne (agrégée) dans chaque pixel coarse
    corresponde à coarse_target_2d.

    - UNIFORM: correction constante par cellule coarse
    - WEIGHTED_A: correction redistribuée dans la cellule coarse selon A2d (hydro-cohérent)
                 tout en conservant EXACTEMENT la moyenne coarse.

    Retour: pred_corr_2d, delta_coarse_2d
    """
    pred = pred_fine_2d.astype(np.float32)

    if mask_pred_fine is None:
        mask_valid = np.isfinite(pred)
    else:
        mask_valid = mask_pred_fine.astype(bool) & np.isfinite(pred)

    pred0 = np.where(mask_valid, pred, 0.0).astype(np.float32)
    m01   = mask_valid.astype(np.float32)

    frac = reproject_to_profile(
        m01, fine_profile, None, coarse_profile,
        dst_nodata=np.nan, resampling=Resampling.average
    ).astype(np.float32)

    mean_all = reproject_to_profile(
        pred0, fine_profile, None, coarse_profile,
        dst_nodata=np.nan, resampling=Resampling.average
    ).astype(np.float32)

    mean_valid = mean_all / (frac + eps)

    ct = coarse_target_2d.astype(np.float32)
    valid_ct = np.isfinite(ct)

    delta = np.full_like(ct, np.nan, dtype=np.float32)
    delta[valid_ct] = ct[valid_ct] - mean_valid[valid_ct]

    delta_up = reproject_to_profile(
        np.where(np.isfinite(delta), delta, 0.0).astype(np.float32),
        coarse_profile, None, fine_profile,
        dst_nodata=np.nan, resampling=Resampling.nearest
    ).astype(np.float32)

    if str(mode).upper() == "WEIGHTED_A":
        if A2d is None:
            w = np.ones_like(pred, dtype=np.float32)
        else:
            w = A2d.astype(np.float32)

        w = np.where(mask_valid & np.isfinite(w), w, 0.0).astype(np.float32)

        w_all = reproject_to_profile(
            w, fine_profile, None, coarse_profile,
            dst_nodata=np.nan, resampling=Resampling.average
        ).astype(np.float32)
        w_mean_valid = w_all / (frac + eps)

        w_mean_up = reproject_to_profile(
            np.where(np.isfinite(w_mean_valid), w_mean_valid, 1.0).astype(np.float32),
            coarse_profile, None, fine_profile,
            dst_nodata=np.nan, resampling=Resampling.nearest
        ).astype(np.float32)

        ratio = w / (w_mean_up + eps)
        ratio = np.clip(ratio, float(ratio_clip[0]), float(ratio_clip[1])).astype(np.float32)
        corr = delta_up * ratio
    else:
        corr = delta_up

    pred_corr = pred.copy()
    pred_corr[mask_valid] = pred_corr[mask_valid] + corr[mask_valid]
    pred_corr[~mask_valid] = np.nan

    return pred_corr.astype(np.float32), delta.astype(np.float32)

# [SANS XAI] export_xai_maps retiré

## 9. Pipeline principal — définition de `main()`

Cette cellule ne fait que définir la fonction ; le calcul est lancé dans la cellule suivante.

In [ ]:
# =============================================================================
# 9) MAIN (corrigé)
# =============================================================================
def main():
    global FS_ALWAYS_KEEP

    SR_OUTPUT_START = date(2020, 1, 1)
    SR_OUTPUT_END   = date(2025, 10, 30)

    print("=== PARAMETRES ===")
    print(" BASE_DIR:", BASE_DIR)
    print(" NAME_FILE:", NAME_FILE)
    print(" OUT_DIR:", OUT_DIR)
    print(" TRAIN:", TRAIN_START, "->", TRAIN_END)
    print(" FORECAST:", FORECAST_START, "->", FORECAST_END)
    print(" SR_OUTPUT:", SR_OUTPUT_START, "->", SR_OUTPUT_END)
    print(" FUSION_ENABLE:", FUSION_ENABLE, "| FUSION_ROOT:", FUSION_ROOT)
    print(" HADA:", HADA_ENABLE, "| FE:", FE_ENABLE, "| FS:", FS_MODE, "TabNet=", _HAS_TABNET)
    print("==================\n")

    if USE_SIN_COS_DOY:
        FS_ALWAYS_KEEP = ["sinDOY", "cosDOY"]

    def _unique_path(path: str) -> str:
        if not os.path.exists(path):
            return path
        root, ext = os.path.splitext(path)
        k = 2
        while True:
            cand = f"{root}_v{k}{ext}"
            if not os.path.exists(cand):
                return cand
            k += 1

    # =========================
    # 1) Lire name_files + FUSION
    # =========================
    sections_raw = parse_name_file(NAME_FILE)

    sections, name_file_used = fuse_sections_to_fused_dir(sections_raw)
    print("➡️ name_file utilisé:", name_file_used)

    # dyn dicts
    chirps_dict = build_date_to_abs_path_dict(sections, SECTION_CHIRPS)
    api_dict    = build_date_to_abs_path_dict(sections, SECTION_API)
    gws_dict    = build_date_to_abs_path_dict(sections, SECTION_GWS)
    rzsm_dict   = build_date_to_abs_path_dict(sections, SECTION_RZSM) if _find_section_key_fuzzy(sections, SECTION_RZSM) is not None else {}

    # statics paths (robuste)
    slope_path = get_single_static_path(sections, SECTION_SLOPE)
    dem_path   = get_single_static_path(sections, SECTION_DTM)
    twi_path   = get_single_static_path(sections, SECTION_TWI)
    dtr_path   = get_single_static_path(sections, SECTION_DTR)

    print("\n=== RESOLUTION STATIQUES ===")
    print(" Slope:", slope_path)
    print(" DEM  :", dem_path)
    print(" TWI  :", twi_path)
    print(" DTR  :", dtr_path)
    print("============================\n")

    for lbl, p in [("Slope", slope_path), ("DEM", dem_path), ("TWI", twi_path), ("DTR", dtr_path)]:
        if p is None:
            raise FileNotFoundError(f"❌ Statique manquant [{lbl}] dans {name_file_used} ou fichier introuvable.")

    if len(chirps_dict) == 0 or len(api_dict) == 0 or len(gws_dict) == 0:
        raise RuntimeError("❌ Dictionnaires dates vides: vérifier CHIRPS/API/GWS (après fusion).")

    all_dates = sorted(set(chirps_dict.keys()) & set(api_dict.keys()) & set(gws_dict.keys()))
    if len(all_dates) == 0:
        raise RuntimeError("❌ Aucune date commune CHIRPS/API/GWS (après fusion).")

    date_candidates = [d for d in all_dates if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)]
    if len(date_candidates) == 0:
        raise RuntimeError("❌ Aucune date dans la fenêtre TRAIN après filtrage.")

    train_dates, test_dates = train_test_split(
        date_candidates, test_size=TEST_FRACTION, random_state=RANDOM_SEED, shuffle=True
    )
    train_dates = sorted(train_dates)
    test_dates  = sorted(test_dates)

    print(f"TRAIN n_dates={len(train_dates)} | TEST n_dates={len(test_dates)}\n")

    # =========================
    # 2) Templates fine/coarse + statiques reprojetées + graphe D8
    # =========================
    d0 = train_dates[0]
    _, chirps_prof0, _ = load_raster(chirps_dict[d0])
    _, gws_prof0, _    = load_raster(gws_dict[d0])

    fine_profile   = chirps_prof0
    coarse_profile = gws_prof0

    twi_f   = ensure_static_on_fine(twi_path, fine_profile, "TWI")
    dtr_f   = ensure_static_on_fine(dtr_path, fine_profile, "DTR")
    dem_f   = ensure_static_on_fine(dem_path, fine_profile, "DEM")
    slope_f = ensure_static_on_fine(slope_path, fine_profile, "SLOPE")

    graph = ensure_hydro_graph_fine(dem_f, fine_profile)

    # =========================================================
    # 3) BUILD DATASET (COARSE)
    # =========================================================
    X_train_list, y_train_list, w_train_list = [], [], []
    X_test_list,  y_test_list  = [], []

    feat_names_global = None
    raw_names_global  = None

    def build_Xy_coarse_for_date(d: date, role="train", use_y_mask=True):
        nonlocal feat_names_global, raw_names_global

        tag = d.strftime("%Y%m%d")

        ch, ch_p, ch_n = load_raster(chirps_dict[d])
        ap, ap_p, ap_n = load_raster(api_dict[d])
        ch = to_float_with_nan(ch, ch_n)
        ap = to_float_with_nan(ap, ap_n)

        rz = None
        if d in rzsm_dict:
            rz0, rz_p, rz_n = load_raster(rzsm_dict[d])
            rz = to_float_with_nan(rz0, rz_n)
            rz = reproject_to_profile(rz, rz_p, rz_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)

        ch_f = reproject_to_profile(ch, ch_p, ch_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)
        ap_f = reproject_to_profile(ap, ap_p, ap_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)

        update_region_mask_fine(ch_f, ap_f, rz)
        region_f = get_region_mask_fine(fine_profile["height"], fine_profile["width"])
        mask_valid_fine = region_f.copy()

        if HADA_ENABLE:
            A2d = build_hada_attention(twi_f, dtr_f, ap_f, ch_f, slope_f)
            ap_g = ap_f * (1.0 + float(HADA_GAMMA) * A2d)
            ch_g = ch_f * (1.0 + float(HADA_GAMMA) * A2d)
        else:
            planes_for_att = [ap_f, ch_f]
            if rz is not None:
                planes_for_att.append(rz)
            A2d = build_attention_map_simple(planes_for_att, use_grad=FE_ATT_USE_GRAD)
            ap_g = ap_f
            ch_g = ch_f

        raw_planes = [ch_g.astype(np.float32), ap_g.astype(np.float32)]
        raw_names  = ["CHIRPS", "API"]
        if rz is not None:
            raw_planes.append(rz.astype(np.float32))
            raw_names.append("RZSM")

        raw_planes += [twi_f, dtr_f, dem_f, slope_f]
        raw_names  += ["TWI", "DTR", "DEM", "SLOPE"]

        raw_planes, raw_names = maybe_add_sin_cos(raw_planes, raw_names, fine_profile, d)
        raw_names_global = raw_names

        feat_planes_fine, feat_names = build_feature_stack_hydro(
            raw_planes_2d=raw_planes,
            raw_names=raw_names,
            enable=FE_ENABLE,
            A2d=A2d,
            graph=graph,
            mask_valid_fine=mask_valid_fine
        )

        if feat_names_global is None:
            feat_names_global = feat_names
            with open(os.path.join(OUT_DIR, "feature_names_all.txt"), "w", encoding="utf-8") as f:
                for n in feat_names_global:
                    f.write(n + "\n")

        feat_planes_coarse = build_coarse_features_from_fine_planes(feat_planes_fine, fine_profile, coarse_profile)

        y0, y_p, y_n = load_raster(gws_dict[d])
        y0 = to_float_with_nan(y0, y_n)
        if (y_p["height"] != coarse_profile["height"]) or (y_p["width"] != coarse_profile["width"]):
            y0 = reproject_to_profile(y0, y_p, y_n, coarse_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)
        y2d = y0.astype(np.float32)

        if use_y_mask:
            mask_keep_2d = np.isfinite(y2d)
        else:
            m = np.ones_like(y2d, dtype=bool)
            for cp in feat_planes_coarse:
                m &= np.isfinite(cp)
            mask_keep_2d = m

        idx_flat_mask_keep = np.flatnonzero(mask_keep_2d.ravel())
        if idx_flat_mask_keep.size == 0:
            return None, None, None, None, None, None

        Xcols = [cp.ravel()[idx_flat_mask_keep] for cp in feat_planes_coarse]
        Xc = np.stack(Xcols, axis=1).astype(np.float32)

        if use_y_mask:
            yc = y2d.ravel()[idx_flat_mask_keep].astype(np.float32)
            base_row_mask = np.isfinite(yc)
        else:
            yc = None
            base_row_mask = np.ones((Xc.shape[0],), dtype=bool)

        row_mask = base_row_mask & np.isfinite(Xc).all(axis=1)
        if not np.any(row_mask):
            return None, None, None, None, None, None

        Xc = Xc[row_mask]
        if yc is not None:
            yc = yc[row_mask]

        idx_flat_keep = idx_flat_mask_keep[row_mask].astype(np.int64)

        w = None
        if HADA_ENABLE:
            A_coarse = reproject_to_profile(
                np.where(np.isfinite(A2d), A2d, 0.0).astype(np.float32),
                fine_profile, None, coarse_profile,
                dst_nodata=np.nan, resampling=Resampling.average
            )
            w0 = A_coarse.ravel()[idx_flat_mask_keep][row_mask].astype(np.float32)
            w = (float(HADA_WMIN) + (float(HADA_WMAX) - float(HADA_WMIN)) * w0).astype(np.float32)
            w = np.clip(w, float(HADA_WMIN), float(HADA_WMAX)).astype(np.float32)
            w = np.where(np.isfinite(w), w, 1.0).astype(np.float32)

        if DEBUG_ENABLE and (
            (role == "train" and train_dates.index(d) < DEBUG_ALWAYS_FIRST_N_DATES)
            or (Xc.shape[0] < DEBUG_TRIGGER_NVALID)
        ):
            dyn_dbg = {"CHIRPS": ch_f, "API": ap_f}
            if rz is not None:
                dyn_dbg["RZSM"] = rz
            stat_dbg = {"TWI": twi_f, "DTR": dtr_f, "DEM": dem_f, "SLOPE": slope_f}
            debug_report_date(
                tag, fine_profile, coarse_profile, y2d, dyn_dbg, stat_dbg,
                feat_names, mask_keep_2d,
                extra_msg=f"| role={role} | n_valid_rows={Xc.shape[0]} | use_y_mask={use_y_mask}"
            )

        print(f" ✔️ [{role}] {d} : {Xc.shape[0]} pixels coarse (use_y_mask={use_y_mask})")
        return Xc, yc, w, feat_names, idx_flat_keep, mask_keep_2d

    def build_X_coarse_for_date(d: date, role="coarse_pred"):
        ch, ch_p, ch_n = load_raster(chirps_dict[d])
        ap, ap_p, ap_n = load_raster(api_dict[d])
        ch = to_float_with_nan(ch, ch_n)
        ap = to_float_with_nan(ap, ap_n)

        rz = None
        if d in rzsm_dict:
            rz0, rz_p, rz_n = load_raster(rzsm_dict[d])
            rz = to_float_with_nan(rz0, rz_n)
            rz = reproject_to_profile(rz, rz_p, rz_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)

        ch_f = reproject_to_profile(ch, ch_p, ch_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)
        ap_f = reproject_to_profile(ap, ap_p, ap_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)

        update_region_mask_fine(ch_f, ap_f, rz)
        region_f = get_region_mask_fine(fine_profile["height"], fine_profile["width"])
        mask_valid_fine = region_f.copy()

        if HADA_ENABLE:
            A2d = build_hada_attention(twi_f, dtr_f, ap_f, ch_f, slope_f)
            ap_g = ap_f * (1.0 + float(HADA_GAMMA) * A2d)
            ch_g = ch_f * (1.0 + float(HADA_GAMMA) * A2d)
        else:
            planes_for_att = [ap_f, ch_f]
            if rz is not None:
                planes_for_att.append(rz)
            A2d = build_attention_map_simple(planes_for_att, use_grad=FE_ATT_USE_GRAD)
            ap_g = ap_f
            ch_g = ch_f

        raw_planes = [ch_g.astype(np.float32), ap_g.astype(np.float32)]
        raw_names  = ["CHIRPS", "API"]
        if rz is not None:
            raw_planes.append(rz.astype(np.float32))
            raw_names.append("RZSM")

        raw_planes += [twi_f, dtr_f, dem_f, slope_f]
        raw_names  += ["TWI", "DTR", "DEM", "SLOPE"]
        raw_planes, raw_names = maybe_add_sin_cos(raw_planes, raw_names, fine_profile, d)

        feat_planes_fine, _ = build_feature_stack_hydro(
            raw_planes_2d=raw_planes,
            raw_names=raw_names,
            enable=FE_ENABLE,
            A2d=A2d,
            graph=graph,
            mask_valid_fine=mask_valid_fine
        )

        feat_planes_coarse = build_coarse_features_from_fine_planes(feat_planes_fine, fine_profile, coarse_profile)

        Hc = coarse_profile["height"]
        Wc = coarse_profile["width"]
        m = np.ones((Hc, Wc), dtype=bool)
        for cp in feat_planes_coarse:
            m &= np.isfinite(cp)

        idx_flat = np.flatnonzero(m.ravel())
        if idx_flat.size == 0:
            return None, None, None

        Xcols = [cp.ravel()[idx_flat] for cp in feat_planes_coarse]
        Xc = np.stack(Xcols, axis=1).astype(np.float32)

        row_mask = np.isfinite(Xc).all(axis=1)
        if not np.any(row_mask):
            return None, None, None

        Xc = Xc[row_mask]
        idx_keep = idx_flat[row_mask].astype(np.int64)
        print(f" ✔️ [{role}] {d} : {Xc.shape[0]} pixels coarse (no-y)")
        return Xc, idx_keep, m

    # IMPORTANT: utilise rf_model/selected_feat_idx après entraînement
    def get_coarse_target_for_date(d: date):
        # (A) HIST : coarse observé (GWS)
        if d in gws_dict:
            y0, y_p, y_n = load_raster(gws_dict[d])
            y0 = to_float_with_nan(y0, y_n)
            if (y_p["height"] != coarse_profile["height"]) or (y_p["width"] != coarse_profile["width"]):
                y0 = reproject_to_profile(
                    y0, y_p, y_n, coarse_profile,
                    dst_nodata=np.nan, resampling=Resampling.bilinear
                )
            return y0.astype(np.float32)

        # (B) FORECAST : coarse cible = coarse prédite par le RF
        Xc, idx_keep, _ = build_X_coarse_for_date(d, role="coarse_target_forecast")
        if Xc is None or idx_keep is None or len(idx_keep) == 0:
            return None

        Xc_fs = Xc[:, selected_feat_idx]
        predc = predict_chunked(rf_model, Xc_fs, chunk=200000)

        Hc = coarse_profile["height"]
        Wc = coarse_profile["width"]
        full = np.full((Hc * Wc,), np.nan, dtype=np.float32)
        full[np.asarray(idx_keep, dtype=np.int64)] = predc.astype(np.float32)
        return full.reshape(Hc, Wc).astype(np.float32)

    # =========================
    # 4) Construire train/test coarse
    # =========================
    for d in train_dates:
        Xc, yc, w, _, _, _ = build_Xy_coarse_for_date(d, role="train", use_y_mask=True)
        if Xc is None:
            continue
        X_train_list.append(Xc)
        y_train_list.append(yc)
        if w is not None:
            w_train_list.append(w)

    for d in test_dates:
        Xc, yc, _, _, _, _ = build_Xy_coarse_for_date(d, role="test", use_y_mask=True)
        if Xc is None:
            continue
        X_test_list.append(Xc)
        y_test_list.append(yc)

    if len(X_train_list) == 0 or len(X_test_list) == 0:
        raise RuntimeError("❌ Dataset vide (train/test). Vérifier les données et masques.")

    X_train = np.vstack(X_train_list).astype(np.float32)
    y_train = np.concatenate(y_train_list).astype(np.float32)
    X_test  = np.vstack(X_test_list).astype(np.float32)
    y_test  = np.concatenate(y_test_list).astype(np.float32)

    w_train = None
    if HADA_ENABLE and len(w_train_list) == len(X_train_list):
        w_train = np.concatenate(w_train_list).astype(np.float32)
        w_train = np.where(np.isfinite(w_train), w_train, 1.0).astype(np.float32)
        w_train = np.clip(w_train, float(HADA_WMIN), float(HADA_WMAX)).astype(np.float32)

    FEATURE_NAMES_ALL = feat_names_global
    if FEATURE_NAMES_ALL is None:
        raise RuntimeError("❌ feature_names_global est None (aucune date n'a produit de features).")

    print("\n=== RESUME DATASET ===")
    print(" X_train:", X_train.shape, "| y_train:", y_train.shape, "| weights:", None if w_train is None else w_train.shape)
    print(" X_test :", X_test.shape,  "| y_test :", y_test.shape)
    print(" n_features:", len(FEATURE_NAMES_ALL))
    print("======================\n")

    X_tr, X_val, y_tr, y_val, idx_tr, idx_val = train_test_split(
        X_train, y_train, np.arange(X_train.shape[0]),
        test_size=0.2, random_state=42, shuffle=True
    )
    w_tr = w_train[idx_tr] if (w_train is not None) else None

    # =========================
    # 5) Choix régularisation RF + Feature Selection
    # =========================
    if REG_MODE.upper() == "AUTO":
        rf_params_used = choose_regularized_params(X_tr, y_tr, X_val, y_val, w_tr=w_tr)
    else:
        fixed = {}
        fixed.update(RF_BASE_PARAMS)
        fixed.update(RF_FIXED_REG_PARAMS)
        rf_params_used = _supported_params_only(fixed)

    selected_feat_idx = list(range(len(FEATURE_NAMES_ALL)))
    selected_feat_names = FEATURE_NAMES_ALL[:]

    n_raw = len(raw_names_global) if raw_names_global is not None else 1

    if FS_MODE.upper() == "AUTO":
        selected_feat_idx, selected_feat_names = run_feature_selection(
            rf_params_used, X_tr, y_tr, X_val, y_val,
            FEATURE_NAMES_ALL, n_raw=n_raw, w_tr=w_tr
        )
        with open(os.path.join(OUT_DIR, "selected_features.txt"), "w", encoding="utf-8") as f:
            f.write("Selected feature indices:\n")
            f.write(str(selected_feat_idx) + "\n\nSelected feature names:\n")
            for n in selected_feat_names:
                f.write(n + "\n")

    # =========================
    # 6) Fit modèle RF final
    # =========================
    rf_model = RandomForestRegressor(**rf_params_used, n_jobs=-1, random_state=42, oob_score=True)

    X_train_fs = X_train[:, selected_feat_idx]
    X_test_fs  = X_test[:,  selected_feat_idx]

    t0 = time.time()
    try:
        rf_model.fit(X_train_fs, y_train, sample_weight=w_train)
    except TypeError:
        rf_model.fit(X_train_fs, y_train)
    t1 = time.time()

    ytr_pred = rf_model.predict(X_train_fs)
    yte_pred = rf_model.predict(X_test_fs)

    print("\n╔══════════════════════════════════════════════════════════════╗")
    print("║  MODÈLE RF (COARSE TRAIN) + FE(Hydro) + FS(TabNet/Perm)     ║")
    print("╠══════════════════════════════════════════════════════════════╣")
    print(f"║ fit_ms={(t1-t0)*1000:.2f}")
    print(f"║ features sélectionnées: {len(selected_feat_idx)} / {len(FEATURE_NAMES_ALL)}")
    print(f"║ R² TRAIN={r2_score(y_train, ytr_pred):.4f} | R² TEST={r2_score(y_test, yte_pred):.4f}")
    print("╚══════════════════════════════════════════════════════════════╝\n")

    # =========================================================
    # 7) EXPORT COARSE
    # =========================================================
    def export_coarse_pred_for_dates(date_list, prefix, use_y_mask_for_build=True):
        if not EXPORT_COARSE_ENABLE or (not date_list):
            return

        print(f"\n=== EXPORT COARSE ({prefix}) n_dates={len(date_list)} ===")
        Hc = coarse_profile["height"]
        Wc = coarse_profile["width"]
        Nc = Hc * Wc

        for d in date_list:
            tag = d.strftime("%Y%m%d")

            if use_y_mask_for_build:
                Xc, _, _, _, idx_keep, _ = build_Xy_coarse_for_date(d, role=f"coarse_{prefix}", use_y_mask=True)
            else:
                Xc, idx_keep, _ = build_X_coarse_for_date(d, role=f"coarse_{prefix}")

            if Xc is None or idx_keep is None or len(idx_keep) == 0:
                print(f" ⚠️ {d} : pas de pixels coarse valides à exporter.")
                continue

            Xc_fs = Xc[:, selected_feat_idx]
            pred = predict_chunked(rf_model, Xc_fs, chunk=200000)

            full = np.full((Nc,), np.nan, dtype=np.float32)
            full[np.asarray(idx_keep, dtype=np.int64)] = pred.astype(np.float32)

            out_path = os.path.join(COARSE_DIR, f"GWS_PRED_COARSE_{prefix}_{tag}.tif")
            out_path = _unique_path(out_path)
            write_geotiff(out_path, full.reshape(Hc, Wc), coarse_profile, nodata_value=OUT_NODATA, compress="DEFLATE")
            print(f" ✅ COARSE {prefix} {d} -> {os.path.basename(out_path)}")

    if EXPORT_COARSE_ENABLE:
        if EXPORT_COARSE_TRAIN:
            export_coarse_pred_for_dates(train_dates, prefix="TRAIN",
                                         use_y_mask_for_build=EXPORT_COARSE_USE_Y_MASK_FOR_TRAIN_TEST)
        if EXPORT_COARSE_TEST:
            export_coarse_pred_for_dates(test_dates, prefix="TEST",
                                         use_y_mask_for_build=EXPORT_COARSE_USE_Y_MASK_FOR_TRAIN_TEST)

    # =========================================================
    # 8) MAPPING SR (fine)
    # =========================================================
    def build_X_fine_for_date(d: date):
        ch, ch_p, ch_n = load_raster(chirps_dict[d])
        ap, ap_p, ap_n = load_raster(api_dict[d])
        ch = to_float_with_nan(ch, ch_n)
        ap = to_float_with_nan(ap, ap_n)

        rz = None
        if d in rzsm_dict:
            rz0, rz_p, rz_n = load_raster(rzsm_dict[d])
            rz = to_float_with_nan(rz0, rz_n)
            rz = reproject_to_profile(rz, rz_p, rz_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)

        ch_f = reproject_to_profile(ch, ch_p, ch_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)
        ap_f = reproject_to_profile(ap, ap_p, ap_n, fine_profile, dst_nodata=np.nan, resampling=Resampling.bilinear)

        update_region_mask_fine(ch_f, ap_f, rz)
        region_f = get_region_mask_fine(fine_profile["height"], fine_profile["width"])
        mask_pred_fine = region_f.copy()

        if HADA_ENABLE:
            A2d = build_hada_attention(twi_f, dtr_f, ap_f, ch_f, slope_f)
            ap_g = ap_f * (1.0 + float(HADA_GAMMA) * A2d)
            ch_g = ch_f * (1.0 + float(HADA_GAMMA) * A2d)
        else:
            planes_for_att = [ap_f, ch_f]
            if rz is not None:
                planes_for_att.append(rz)
            A2d = build_attention_map_simple(planes_for_att, use_grad=FE_ATT_USE_GRAD)
            ap_g = ap_f
            ch_g = ch_f

        raw_planes = [ch_g.astype(np.float32), ap_g.astype(np.float32)]
        raw_names  = ["CHIRPS", "API"]
        if rz is not None:
            raw_planes.append(rz.astype(np.float32))
            raw_names.append("RZSM")

        raw_planes += [twi_f, dtr_f, dem_f, slope_f]
        raw_names  += ["TWI", "DTR", "DEM", "SLOPE"]

        raw_planes, raw_names = maybe_add_sin_cos(raw_planes, raw_names, fine_profile, d)

        feat_planes_fine, feat_names_full = build_feature_stack_hydro(
            raw_planes_2d=raw_planes,
            raw_names=raw_names,
            enable=FE_ENABLE,
            A2d=A2d,
            graph=graph,
            mask_valid_fine=mask_pred_fine
        )

        mask_flat = mask_pred_fine.ravel()
        idx_pred = np.flatnonzero(mask_flat)

        cols = [fp.ravel()[idx_pred] for fp in feat_planes_fine]
        X_full = np.stack(cols, axis=1).astype(np.float32)

        row_mask = np.isfinite(X_full).all(axis=1)
        if not np.any(row_mask):
            return None, None, None, None, None, None

        idx_keep = idx_pred[row_mask]
        X_keep_full = X_full[row_mask]
        X_keep = X_keep_full[:, selected_feat_idx]
        return X_keep, idx_keep, A2d, (feat_planes_fine, feat_names_full), (raw_planes, raw_names), mask_pred_fine

    def run_mapping_for_dates(date_list, prefix):
        print(f"\n=== SR MAPPING: {prefix} (n_dates={len(date_list)}) ===")

        Hf = int(fine_profile["height"])
        Wf = int(fine_profile["width"])
        Nf = Hf * Wf

        for d in date_list:
            if d in EXCLUDED_DATES:
                continue

            X_keep, idx_keep, A2d, feat_pack, raw_pack, mask_pred_fine = build_X_fine_for_date(d)
            if X_keep is None or idx_keep is None or len(idx_keep) == 0:
                print(f" ⚠️ {d} : pas de pixels valides en SR.")
                continue

            pred = predict_chunked(rf_model, X_keep, chunk=200000).astype(np.float32)

            pred_full = np.full((Nf,), np.nan, dtype=np.float32)
            pred_full[np.asarray(idx_keep, dtype=np.int64)] = pred
            pred_2d = pred_full.reshape(Hf, Wf)

            if MS_ENABLE:
                coarse_target = None
                try:
                    coarse_target = get_coarse_target_for_date(d)
                except Exception as e:
                    print(f" ⚠️ {d} : get_coarse_target_for_date a échoué ({e})")
                    coarse_target = None

                if coarse_target is None:
                    print(f" ⚠️ {d} : coarse_target indisponible -> Multi-scale consistency skip.")
                else:
                    try:
                        pred_2d_corr, delta_coarse = enforce_multiscale_consistency(
                            pred_fine_2d=pred_2d,
                            fine_profile=fine_profile,
                            coarse_target_2d=coarse_target,
                            coarse_profile=coarse_profile,
                            mask_pred_fine=mask_pred_fine,
                            A2d=A2d,
                            mode=MS_MODE,
                            eps=MS_EPS,
                            ratio_clip=(MS_RATIO_CLIP_MIN, MS_RATIO_CLIP_MAX),
                        )
                        pred_2d = pred_2d_corr

                        if MS_EXPORT_DIAG:
                            tag = d.strftime("%Y%m%d")
                            diag_path = os.path.join(MS_DIAG_DIR, f"MS_DeltaCoarse_{prefix}_{tag}.tif")
                            diag_path = _unique_path(diag_path)
                            write_geotiff(
                                diag_path,
                                delta_coarse.astype(np.float32),
                                coarse_profile,
                                nodata_value=OUT_NODATA,
                                compress="DEFLATE",
                            )
                    except Exception as e:
                        print(f" ⚠️ {d} : erreur Multi-scale consistency -> skip. ({e})")

            tag = d.strftime("%Y%m%d")
            suffix = "_MS" if MS_ENABLE else ""
            out_path = os.path.join(OUT_DIR, f"SR_GWS_RF_{prefix}_{tag}{suffix}.tif")
            out_path = _unique_path(out_path)
            write_geotiff(out_path, pred_2d, fine_profile, nodata_value=OUT_NODATA, compress="DEFLATE")
            print(f" ✅ {d} -> {os.path.basename(out_path)}")
            # [SANS XAI] incertitude par arbre, export_xai_maps et HydroINV retirés ici

    # =========================================================
    # 9) SR OUTPUT: 2020-01-01 -> 2025-10-30
    # =========================================================
    sr_dates = [
        d for d in date_range(SR_OUTPUT_START, SR_OUTPUT_END)
        if (d in chirps_dict) and (d in api_dict) and (d not in EXCLUDED_DATES)
    ]
    if not sr_dates:
        raise RuntimeError("❌ Aucune date SR candidate dans [SR_OUTPUT_START, SR_OUTPUT_END] avec CHIRPS+API.")

    hist_dates     = [d for d in sr_dates if d in gws_dict]
    forecast_dates = [d for d in sr_dates if d not in gws_dict]

    print(f"\n=== SR OUTPUT SPLIT ===")
    print(f" HIST dates    : {len(hist_dates)} (avec GWS coarse existant)")
    print(f" FORECAST dates: {len(forecast_dates)} (sans GWS coarse -> coarse forecast possible)")
    print("=======================\n")

    if EXPORT_COARSE_ENABLE and EXPORT_COARSE_FORECAST and forecast_dates:
        export_coarse_pred_for_dates(
            forecast_dates,
            prefix="FORECAST",
            use_y_mask_for_build=False
        )

    if hist_dates:
        run_mapping_for_dates(hist_dates, prefix="HIST")
    if forecast_dates:
        run_mapping_for_dates(forecast_dates, prefix="FORECAST")

    print("\n🎉 Terminé : (FUSION inputs) + SR_GWS + HADA + Hydro-FE + FS (sans XAI).")

## 10. Exécution

Lance toute la chaîne : fusion → dataset coarse → RF + sélection de variables → export coarse → mapping SR.

In [ ]:
main()

## 11. Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers produits par dossier de sortie
for _lbl, _d, _rec in [("SR (fine)", OUT_DIR, False), ("Coarse", COARSE_DIR, True)]:
    _sub = ("**",) if _rec else ()
    _n_tif = len(glob.glob(os.path.join(_d, *_sub, "*.tif"), recursive=_rec))
    _n_csv = len(glob.glob(os.path.join(_d, *_sub, "*.csv"), recursive=_rec))
    print(f"{_lbl:10s} {_n_tif:6d} .tif | {_n_csv:4d} .csv  ->  {_d}")